# `data.pt` sanity check for signal leak between trials (XCP-D reprocessing)

Loads one subject's `data.pt` blob (produced by `preprocessing_volume.py`), dequantizes it, and
runs a series of checks to confirm the XCP-D-based reprocessing looks as expected:

1. Shape / scale / dtype sanity
2. Parcellate with schaefer400 and check the signal for each imagery trial
**Run this on the Linux machine** (needs `torch`, `numpy`, `matplotlib`, and the real data) --
not on a machine without the data mounted. Edit `DATA_PT_PATH` (and `FMRIPREP_DATA_PT_PATH` in the
last section) below to point at your actual files before running.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

: 

## 1. Load and dequantize

`data.pt` stores `{"frames": int8 tensor [T,H,W,D], "scale": float, "num_frames": int}`.
Dequantize with `frames.to(torch.float32) * scale`.

In [ ]:
# EDIT ME: path to the subject's data.pt under the newly-generated XCP-D output tree
DATA_PT_PATH = "/nfs/turbo/umms-huanglab/mkersey/Michigan_Dataset_1_for_NeuroSTORM/FIXED_xcp_nosmooth_task-imagery_run-1/img/sub-04/data.pt"
TASK_TIMING_PATH = "/nfs/turbo/umms-huanglab/mkersey/Michigan_Dataset_1_for_NeuroSTORM/FIXED_xcp_nosmooth_task-imagery_run-1/metadata/timing_onehot_task1.tsv"

blob = torch.load(DATA_PT_PATH, map_location="cpu", weights_only=False)
frames_int8 = blob["frames"]
scale = float(blob["scale"])
num_frames = int(blob["num_frames"])

print("frames dtype:", frames_int8.dtype)
print("frames shape (T, H, W, D):", tuple(frames_int8.shape))
print("scale:", scale)
print("num_frames (recorded):", num_frames, " matches shape[0]:", frames_int8.shape[0] == num_frames)

data = frames_int8.to(torch.float32) * scale
print("dequantized min/max:", data.min().item(), data.max().item())
print("dequantized mean/std (all voxels, all frames):", data.mean().item(), data.std().item())

frames dtype: torch.int8
frames shape (T, H, W, D): (1145, 96, 96, 96)
scale: 0.2762128875011534
num_frames (recorded): 1145  matches shape[0]: True


## (Optional) Save dequantized data.pt as a nifti file so you can examine it in MRIcrogl

In [ ]:
# # Convert data to nifti file and save
# import nibabel as nib
# data_np = data.numpy()
# nib.save(nib.Nifti1Image(data_np, affine=None), "~/Downloads/data.nii.gz")

## 2. Background-corner check

A corner block (`[:5, :5, :5]`) should be pure background (outside the brain). Its temporal std
should be ~0 (a fixed fill value every frame) regardless of what that fill value actually is
(recall `fill_zeroback=False` by default upstream, so it may not literally be 0).

In [ ]:
corner = data[:, :5, :5, :5]
corner_vals = torch.unique(corner)
print("distinct values in an assumed-background corner:", corner_vals.numel())
print("those values:", corner_vals[:10].tolist())

corner_time_std = corner.std(dim=0)
print("max per-voxel temporal std in that corner (should be ~0):", corner_time_std.max().item())

## 3. Mid-slice visualization (single frame)

Sagittal / coronal / axial mid-slices of one frame -- a quick look for gross artifacts, wrong
orientation, or a mangled resample.

In [ ]:
frame_index = 0
vol = data[frame_index]
H, W, D = vol.shape

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.subplots_adjust(right=0.84, wspace=0.15)  # leave space for the colorbar

vmin, vmax = np.percentile(data.numpy(), [1, 99])

im0 = axes[0].imshow(vol[H // 2, :, :].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
axes[0].set_title(f"Sagittal (frame {frame_index})")

im1 = axes[1].imshow(vol[:, W // 2, :].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
axes[1].set_title(f"Coronal (frame {frame_index})")

im2 = axes[2].imshow(vol[:, :, D // 2].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
axes[2].set_title(f"Axial (frame {frame_index})")

for ax in axes:
    ax.axis("off")

# Create a dedicated colorbar axes outside the subplots so it does not overlap
cbar_ax = fig.add_axes([0.87, 0.12, 0.02, 0.76])
sm = plt.cm.ScalarMappable(cmap="gray", norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.ax.set_ylabel("intensity", rotation=90, va="center")

plt.show()

## 4. Per-voxel temporal mean ("fingerprint") map

This is the value that directly answers the XCP-D-demeaning question: if XCP-D's confound
regression (with its intercept term) plus high-pass filtering already drove each voxel's temporal
mean toward ~0, this map should look faint/near-zero almost everywhere in the foreground --
unlike an un-regressed fMRIPrep map, where anatomical structure typically shows through clearly
(the "fingerprint" `fix_voxelwise_normalization.py` targets).

In [ ]:
voxel_time_mean = data.mean(dim=0)
print("per-voxel temporal mean: min/max/mean-abs:",
      voxel_time_mean.min().item(), voxel_time_mean.max().item(), voxel_time_mean.abs().mean().item())

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(voxel_time_mean[H // 2, :, :].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
axes[0].set_title("Sagittal: per-voxel temporal mean")
axes[1].imshow(voxel_time_mean[:, W // 2, :].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
axes[1].set_title("Coronal: per-voxel temporal mean")
axes[2].imshow(voxel_time_mean[:, :, D // 2].T, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
axes[2].set_title("Axial: per-voxel temporal mean")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Histogram of voxel values 

In [ ]:
# Check whether background voxels are all zero and  their values
frame0 = data[0]
bg_val = frame0.min()
print("frame0 min (bg_val):", float(bg_val))

# Recreate mask exactly as the histogram cell does
mask = fg_mask if ('fg_mask' in globals() and isinstance(fg_mask, torch.Tensor) and fg_mask.any()) else None
if mask is None:
    mask = (frame0 != bg_val)
print("foreground voxels:", int(mask.sum()), "total voxels:", mask.numel())

# Background values across time (may be large; sample unique up to 50)
bg = data[:, ~mask]
print("background shape (T, N_bg):", tuple(bg.shape))
bg_vals = torch.unique(bg)
print("unique background values (count):", int(bg_vals.numel()))
print("background min/max:", float(bg_vals.min()), float(bg_vals.max()))
print("first background values (up to 50):", bg_vals[:50].tolist())

In [ ]:
mask[mask==True].shape

## 5. Single foreground voxel's timeseries

A quick visual check for drift/low-frequency content -- with high-pass filtering on, this should
hover around ~0 with no slow drift.

In [ ]:
cx, cy, cz = H // 2, W // 2, D // 2
voxel_ts = data[:, cx, cy, cz].numpy()

plt.figure(figsize=(10, 3))
plt.plot(voxel_ts)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.title(f"Timeseries at voxel ({cx},{cy},{cz})")
plt.xlabel("frame")
plt.ylabel("dequantized intensity")
plt.show()

print("this voxel's timeseries mean:", voxel_ts.mean(), " std:", voxel_ts.std())

## 6. Mean foreground signal per frame (global signal)

Uses the background-corner value (if it was a single constant, from section 2) to build a
foreground mask from frame 0, then averages over all foreground voxels per frame.

In [ ]:
background_val = corner_vals[0].item() if corner_vals.numel() == 1 else None
if background_val is not None:
    fg_mask = (data[0] != background_val)
    global_signal = data[:, fg_mask].mean(dim=1).numpy()
    plt.figure(figsize=(10, 3))
    plt.plot(global_signal)
    plt.title("Mean foreground signal per frame (global signal)")
    plt.xlabel("frame")
    plt.ylabel("mean intensity")
    plt.show()
else:
    print("Corner block wasn't a single constant value --  corner_vals before assuming background.")

## 7. (Optional) Side-by-side vs. the fMRIPrep-based `data.pt` for the same subject

Directly compares the per-voxel temporal-mean "fingerprint" magnitude between the XCP-D and
fMRIPrep versions of the same subject -- the empirical check for whether XCP-D's confound
regression already shrank the fingerprint that motivated `fix_voxelwise_normalization.py`.
Skip/edit this cell if you don't have (or don't want to compare against) the fMRIPrep version.

In [ ]:
# EDIT ME: path to the SAME subject's data.pt from the original fMRIPrep-based processed_output_v2
FMRIPREP_DATA_PT_PATH = "/data/margo_datasets/Michigan_Dataset_1_for_NeuroSTORM/processed_output_v2/img/sub-04/data.pt"

blob_fp = torch.load(FMRIPREP_DATA_PT_PATH, map_location="cpu", weights_only=False)
data_fp = blob_fp["frames"].to(torch.float32) * float(blob_fp["scale"])
mean_fp = data_fp.mean(dim=0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(voxel_time_mean[:, :, D // 2].T, cmap="gray", origin="lower")
axes[0].set_title("XCP-D: per-voxel temporal mean (axial)")
axes[1].imshow(mean_fp[:, :, mean_fp.shape[2] // 2].T, cmap="gray", origin="lower")
axes[1].set_title("fMRIPrep: per-voxel temporal mean (axial)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("XCP-D  mean|fingerprint|:", voxel_time_mean.abs().mean().item())
print("fMRIPrep mean|fingerprint|:", mean_fp.abs().mean().item())